# Уточняющие вопросы по тестовому кейсу

Ноутбук читает выбранное описание из `data/test` и строит два независимых списка из **7 вопросов в порядке важности**:

1. только по тексту кейса;
2. по тексту кейса с учетом каталога параметров из `output/cost_estimation_parameters.csv`.

Каталог используется как источник идей и примеров: модель должна игнорировать нерелевантные параметры. Первый запрос не получает каталог, поэтому списки можно корректно сравнить.

Перед запуском создайте `.env` с `OPENAI_API_KEY=...`. Модель задается через `OPENAI_MODEL`, а конкретный кейс — через `TEST_CASE_FILE` (имя файла или путь). Если `TEST_CASE_FILE` не задан, используется первый файл в `data/test`.

In [1]:
# При необходимости раскомментируйте:
# %pip install openai python-dotenv pandas

from pathlib import Path
from datetime import datetime, timezone
import json
import math
import os
import random

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

## Настройки и входные данные

Чтобы выбрать кейс прямо в ноутбуке, замените `CASE_FILE` на имя файла из `data/test`, например `"case_082_optimal_drop_times_using_machine_learning.md"`. Абсолютный или существующий относительный путь также поддерживается.

In [2]:
TEST_DIR = Path("data/test")
PARAMETERS_CSV = Path("output/cost_estimation_parameters.csv")
CASE_FILE = os.getenv("TEST_CASE_FILE")  # Например: "case_082_optimal_drop_times_using_machine_learning.md"

load_dotenv()
MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")
MAX_API_ATTEMPTS = 3
QUESTION_COUNT = 7
EVALUATOR_MODEL = os.getenv("OPENAI_EVALUATOR_MODEL", MODEL)
EVALUATION_DIR = Path("output/clarifying_questions_test")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("В .env не найден OPENAI_API_KEY")
if not TEST_DIR.is_dir():
    raise RuntimeError(f"Папка с тестовыми кейсами не найдена: {TEST_DIR}")
if not PARAMETERS_CSV.is_file():
    raise RuntimeError(f"Таблица параметров не найдена: {PARAMETERS_CSV}")

case_paths = sorted(path for path in TEST_DIR.iterdir() if path.is_file())
if not case_paths:
    raise RuntimeError(f"В {TEST_DIR} не найдено ни одного кейса")

if CASE_FILE:
    supplied_path = Path(CASE_FILE)
    case_path = supplied_path if supplied_path.is_file() else TEST_DIR / supplied_path
    if not case_path.is_file():
        available = ", ".join(path.name for path in case_paths)
        raise FileNotFoundError(f"Кейс не найден: {CASE_FILE}. Доступны: {available}")
else:
    case_path = case_paths[0]

case_text = case_path.read_text(encoding="utf-8").strip()
if not case_text:
    raise ValueError(f"Файл кейса пуст: {case_path}")

parameters_df = pd.read_csv(PARAMETERS_CSV, encoding="utf-8-sig").fillna("")
required_columns = {
    "параметр",
    "подробное описание параметра",
    "примеры значений",
    "важность (1-10)",
    "когда применять этот параметр оценки",
    "когда не применять этот параметр",
}
missing_columns = required_columns - set(parameters_df.columns)
if missing_columns:
    raise ValueError(f"В таблице отсутствуют столбцы: {sorted(missing_columns)}")

client = OpenAI()
print(f"Модель: {MODEL}")
print(f"Кейс: {case_path}")
print(f"Символов в описании: {len(case_text)}")
print(f"Параметров в каталоге: {len(parameters_df)}")
print(f"Модель-оценщик: {EVALUATOR_MODEL}")

Модель: gpt-5-mini
Кейс: data/test/case_081_zillow_floor_plan_training_models_to_detect_windows_doors_and_openings_in_panora.md
Символов в описании: 20336
Параметров в каталоге: 75
Модель-оценщик: gpt-5-mini


## Промпты и проверка ответа

Оба режима требуют ровно 7 самостоятельных вопросов. Номер `1` означает самый важный вопрос. Проверка отклоняет ответы с неверным количеством, порядком, пустыми полями или дублями.

In [3]:
SYSTEM_PROMPT = """
Ты — ведущий системный аналитик, который готовит требования для самостоятельной разработки ML-, AI-, data- или backend-решения по описанию референсного кейса.
Сформулируй ровно 7 самых важных уточняющих вопросов, ответы на которые сильнее всего влияют на границы, архитектуру, трудоемкость и критерии приемки подобного решения.

Правила:
1. Пиши только на русском языке.
2. Расположи вопросы по убыванию важности: priority от 1 (самый важный) до 7.
3. Каждый пункт должен быть конкретным вопросом, на который заказчик может дать содержательный ответ. Допускается поясняющая просьба после вопросительного знака, например: «Какие источники доступны? Укажите ограничения для каждого»
4. Не спрашивай то, что уже однозначно указано в кейсе; неизвестные, предположительные и неоднозначные сведения нужно уточнять.
5. Не объединяй несколько независимых тем в один перегруженный вопрос.
6. Избегай дублей и общих вопросов без объяснимого влияния на разработку.
7. В why_important кратко объясни, какое решение или объем работ зависит от ответа.
8. Верни только корректный JSON без Markdown.

Формат: {"questions": [{"priority": 1, "question": "...", "why_important": "..."}]}
""".strip()


def normalize_questions(payload):
    questions = payload.get("questions")
    if not isinstance(questions, list) or len(questions) != QUESTION_COUNT:
        actual = len(questions) if isinstance(questions, list) else "не список"
        raise ValueError(f"Ожидалось {QUESTION_COUNT} вопросов, получено: {actual}")

    normalized = []
    for item in questions:
        if not isinstance(item, dict):
            raise ValueError("Каждый вопрос должен быть JSON-объектом")
        try:
            priority = int(item.get("priority"))
        except (TypeError, ValueError) as error:
            raise ValueError("priority должен быть целым числом") from error
        question = str(item.get("question", "")).strip()
        why_important = str(item.get("why_important", "")).strip()
        if not question or not why_important:
            raise ValueError("У вопроса пустое поле question или why_important")
        if "?" not in question:
            raise ValueError(f"Формулировка должна быть вопросом: {question}")
        normalized.append({
            "priority": priority,
            "question": question,
            "why_important": why_important,
        })

    priorities = [item["priority"] for item in normalized]
    if priorities != list(range(1, QUESTION_COUNT + 1)):
        raise ValueError(f"Неверный порядок priority: {priorities}")
    canonical_questions = [item["question"].casefold() for item in normalized]
    if len(canonical_questions) != len(set(canonical_questions)):
        raise ValueError("В ответе есть одинаковые вопросы")
    return normalized


def request_questions(user_prompt):
    last_error = None
    for attempt in range(1, MAX_API_ATTEMPTS + 1):
        correction = "" if last_error is None else (
            f"\n\nПредыдущий ответ не прошел проверку: {last_error}. Исправь ответ."
        )
        response = client.responses.create(
            model=MODEL,
            instructions=SYSTEM_PROMPT,
            input=user_prompt + correction,
            text={"format": {"type": "json_object"}},
        )
        try:
            return normalize_questions(json.loads(response.output_text))
        except (json.JSONDecodeError, TypeError, ValueError) as error:
            last_error = str(error)
            print(f"Попытка {attempt} не прошла проверку: {last_error}")
    raise RuntimeError(f"Не удалось получить корректный список вопросов: {last_error}")

## 1. Вопросы только по описанию кейса

В этом запросе модель не видит таблицу параметров.

In [4]:
base_user_prompt = f"""
Подготовь уточняющие вопросы для самостоятельной разработки решения, подобного описанному ниже.

<case>
{case_text}
</case>

Верни результат строго в формате JSON.
""".strip()

questions_without_catalog = request_questions(base_user_prompt)

Попытка 1 не прошла проверку: Формулировка должна быть вопросом: Дайте детальные правила классов: что однозначно считается door, window, opening и как обрабатывать спорные кейсы (душевые/раздвижные двери, шкафы, зеркальные отражения, частично видимые/окклюзированные объекты, группы соседних окон — считать одним объектом или несколькими). Приложите, пожалуйста, несколько обязательных положительных и отрицательных примеров для каждого класса


## 2. Вопросы с учетом каталога параметров

В запрос передается полный каталог. Инструкция разрешает использовать только релевантные строки и запрещает искусственно включать параметры ради покрытия таблицы.

In [5]:
catalog_records = parameters_df.to_dict(orient="records")
catalog_json = json.dumps(catalog_records, ensure_ascii=False)

catalog_user_prompt = f"""
Подготовь уточняющие вопросы для самостоятельной разработки решения, подобного описанному ниже.

Используй каталог параметров только как источник возможных идей, формулировок и примеров. Сначала оцени применимость каждого кандидата к конкретному кейсу. Не включай параметр только потому, что он присутствует в каталоге, и не пытайся обеспечить покрытие таблицы. Итог по-прежнему должен содержать только 7 наиболее важных вопросов для этого кейса.

<case>
{case_text}
</case>

<parameters_catalog>
{catalog_json}
</parameters_catalog>

Верни результат строго в формате JSON.
""".strip()

questions_with_catalog = request_questions(catalog_user_prompt)

## 3. Оба списка

In [6]:
def show_questions(title, questions):
    display(Markdown(f"### {title}"))
    frame = pd.DataFrame(questions).rename(columns={
        "priority": "Приоритет",
        "question": "Уточняющий вопрос",
        "why_important": "Почему это важно",
    })
    display(frame.style.hide(axis="index"))


show_questions("Без учета таблицы параметров", questions_without_catalog)
show_questions("С учетом таблицы параметров", questions_with_catalog)

### Без учета таблицы параметров

Приоритет,Уточняющий вопрос,Почему это важно
1,"Дайте детальные правила классов: что однозначно считается ""door"" (дверь), ""window"" (окно) и ""opening"" (проём) и как обрабатывать спорные кейсы (душевые/раздвижные двери, шкафы/кладовые, зеркальные отражения, частично видимые/окклюзированные объекты, группы соседних окон — считать одним объектом или несколькими)? Приложите, пожалуйста, минимум по 3 обязательных положительных и 3 обязательных отрицательных примера для каждого класса.","Однозначные правила классов и примеры критичны для согласованных аннотаций, снижают межаннотаторную вариативность, определяют целевую разметку и напрямую влияют на метрики, архитектуру модели и требования к постобработке."
2,"Какой точный формат аннотаций вы требуете: координаты bbox в пикселях или нормализованные (x_min,x_max,y_min,y_max) относительно equirectangular изображения; нужно ли поддерживать боксы, пересекающие правый/левый край (wrap-around), или использовать альтернативы (полигон, широта/долгота центра + угловая ширина)? Уточните также правило для верхней/нижней границы (насколько «расслаблены» значения, допустимое отклонение) и минимально/максимально допустимые размеры боксов.","Формат разметки и правила для wrap-around/верха-низа определяют реализацию датапайплайна, расчёт IoU, архитектуру выходных слоёв модели, а также сложность обучения/аугментаций и мерджинга предсказаний."
3,"Какие исходные данные и метаданные доступны: общее число панорам, число аннотированных боксов по каждому классу, текущий train/val/test сплит, разрешение изображений, наличие EXIF/pose/высоты камеры/информации о плане этажа или уже подготовленных перспективных кропов; и какая доля панорам содержит occlusion/loop-closure/mirror-case? Приложите таблицу или числа распределения по классам и типам помещений, если есть.","Объём и состав датасета влияют на выбор модели, стратегию борьбы с дисбалансом, требования к аугментациям, необходимость дополнительной разметки и оценку рисков недообучения или переобучения."
4,"Какие точные метрики и пороги приёма вы ожидаете в продакшене: AP@IoU=0.5, mAP@0.5:0.95 или иные; требуемые минимальные значения для каждого класса (door/window/opening); необходимо ли сравнение с человеческой базой (укажите желаемое соотношение к human baseline) и какие trade-off между precision/recall приемлемы (например, минимум precision при recall≥X)?","Целевые метрики определяют как выбирать модель/гиперпараметры, когда считать задачу решённой, какие компромиссы между скоростью и качеством допустимы, и как настраивать thresholding/калибровку."
5,"Какие ограничения по inference/деплойменту: допустимая задержка на одну панораму, целевой throughput (панорам/минуту), аппаратная платформа (серверный GPU, CPU-only, edge/mobilные устройства), ограничения по памяти/размеру модели, необходимость пакетной обработки и допустимость дополнительных стадий (генерация перспективных кропов и последующий фьюзинг)?","Ограничения деплоя критически влияют на выбор семейства моделей (one-stage vs two-stage), необходимость оптимизаций (квантизация, pruning), и на архитектуру препроцессинга/постобработки."
6,"Какой подход к обучению вы предпочитаете и какие допуски возможны: прямое обучение на equirectangular панорамах (без изменений свёрток), использование перспективных кропов с последующим фьюзингом, или адаптация архитектуры (spherical convolutions)? Разрешено ли гибридное решение (train на панорамах + доп. перспективные данные) и сколько дополнительной инженерной/исследовательской работы вы готовы принять ради улучшения качества?","Выбор маршрута обучения определяет объём исследований, сложность реализации (модификация свёрток, фьюзинг), требования к обучающим данным и ожидаемый выигрыш в качестве/латентности."
7,"Какой процесс контроля качества аннотаций и разрешения спорных случаев вы хотите внедрить: требуется ли регулярный коллективный ревью (ежедневно/еженедельно), golden-set для оценки аннотаторов, процедура аджудации спорных примеров, и нужно ли иметь инстр

### С учетом таблицы параметров

Приоритет,Уточняющий вопрос,Почему это важно
1,"Где и как будут выполняться предсказания в продакшене — на сервере в облаке (GPU/CPU), на edge/встраиваемом устройстве (инференс на CPU/GPU/TPU) или гибридно? Укажите ожидаемую пропускную способность (сред./пик запросов в час) и целевые SLA по задержке (например, p50/p95 в миллисекундах/секундах).","Ответ определит выбор модели (Faster‑R-CNN vs SSD vs мобильные варианты), необходимость оптимизаций (quantization, pruning), архитектуру сервиса (централизованный predict‑service vs in‑process), и расчёт infra/cost для inference."
2,"Насколько однородны входные панорамы в продакшене: всегда ли они выровнены (leveled) и предоставляются в equirectangular проекции, каково типичное разрешение и список камер/форматов (например Ricoh Theta, Insta360)? Доступны ли метаданные (yaw/pitch/roll, высота камеры над полом) вместе с изображениями?","Если панорамы не всегда выровнены или имеют разные проекции/разрешения/метаданные, потребуется стадия выравнивания/нормализации, другие предобработки или изменение модели (например perspective crops или сферические свёртки), а также повлияет на точность проекции боксов на план."
3,"Какие точные критерии приёмки модели ожидаются для запуска в production — по каким метрикам и порогам (например AP@IoU=0.5 по классам door/window/opening, минимальные значения для p95 latency), и нужно ли превосходить человеческий уровень или достаточно близости к нему? Укажите отдельно допустимые trade‑offs для FP vs FN (что критичнее для downstream floorplan).","Чёткие целевые метрики влияют на объём тренировочных данных, стратегию валидации, баланс классов, стоимость дообучения и решение о дополнительной ручной проверке/фильтрации предсказаний перед использованием в генераторе планов."
4,"Как будут использоваться предсказанные боксы дальше: автоматически (без ручной проверки) в пайплайне генерации floorplan, или сначала проходят валидацию человеком/правилами? Опишите ожидаемое поведение при низкой уверенности модели (например: отклонять, пометить для ревью, применять fallback‑логику).","Если выводы применяются автоматически, потребуется более жёсткая валидация, тесты и fallback‑механизмы; для ручной валидации допустим более высокий recall и другие UX‑решения. Это напрямую влияет на требования к надежности, monitoringu и рабочему процессу QA."
5,"В каком формате и системе координат нужно отдавать предсказания модели: пиксельные bounding box в equirectangular картинке, угловые координаты (θ,φ), или уже проецированные координаты на план пола (пересечение с плоскостью пола)? Уточните требование по ограничению по вертикали (только L/R tight, T/B relaxed) и единицы измерения/ориентацию.","Формат вывода определяет архитектуру головы детектора, пост‑процессинг (объединение боксов через wrap‑around), необходимость конвертации координат и интеграцию с модулем построения плана — изменения на этом уровне сильно влияют на интерфейс между компонентами и на тесты приёмки."
6,"Есть ли чёткие, окончательные правила разметки для спорных кейсов (occluded doors, shower doors, двери в зеркале, дверные проёмы без дверей, группы рядом стоящих окон — мерджить или разделять)? Если правил нет — готовы ли вы согласовать и переразметить часть датасета?","Последовательность аннотаций критична: неоднозначные правила увеличивают шум в таргетах, ухудшают обучение и приводят к класс‑конфузии (особенно opening). Решение об уточнении/переразметке влияет на объём работ по data‑engineering и качество модели."
7,"Какие ограничения по бюджету/ресурсам на обучение и inference допустимы (максимальный недельный бюджет на обучение, доступ к GPU‑кластеру/TPU и допустимая стоимость на изображение при продакшн‑инференсе)?","Бюджет и доступные вычислительные ресурсы определяют выбор стратегии (тренировать тяжёлую двухступенчатую модель или лёгкую one‑stage), масштаб экспериментов, необходимость оптимизаций для inference и тот объём инженерных усилий, который реалистично заложить в план работ."


## 4. Слепое тестирование на всем тестовом множестве

Для каждого файла из `data/test` заново строятся оба списка. Перед передачей модели-оценщику списки случайно назначаются меткам `A` и `B`, поэтому подход с каталогом не занимает фиксированную позицию. Оценщик видит описание кейса и два списка, но не знает способ их получения, и выбирает более профессиональный список с позиции заказчика.

Результаты сохраняются после каждого оцененного кейса:

- `short_log.csv` — краткий итог (какой подход признан более профессиональным);
- `detailed_log.jsonl` — оба списка, случайный порядок, выбор и комментарий оценщика;
- `summary.json` — агрегаты и односторонний точный биномиальный тест гипотезы о том, что подход с каталогом выигрывает случайно с вероятностью 0.5.


In [7]:
EVALUATOR_SYSTEM_PROMPT = """
Ты — заказчик ML-, AI-, data- или backend-решения. Сравни два списка уточняющих вопросов к описанию проекта вслепую.
Выбери список, который кажется более профессиональным и лучше демонстрирует понимание сути задачи. Учитывай релевантность кейсу, влияние вопросов на границы и стоимость работ, конкретность и отсутствие повторов.
Нельзя объявлять ничью и нельзя догадываться о способе создания списков. Верни только корректный JSON без Markdown.

Формат: {"winner": "A|B", "reason": "содержательное объяснение выбора"}
""".strip()


def make_question_prompts(text):
    base_prompt = f"""
Подготовь уточняющие вопросы для самостоятельной разработки решения, подобного описанному ниже.

<case>
{text}
</case>

Верни результат строго в формате JSON.
""".strip()
    catalog_prompt = f"""
Подготовь уточняющие вопросы для самостоятельной разработки решения, подобного описанному ниже.

Используй каталог параметров только как источник возможных идей, формулировок и примеров. Сначала оцени применимость каждого кандидата к конкретному кейсу. Не включай параметр только потому, что он присутствует в каталоге, и не пытайся обеспечить покрытие таблицы. Итог по-прежнему должен содержать только 7 наиболее важных вопросов для этого кейса.

<case>
{text}
</case>

<parameters_catalog>
{catalog_json}
</parameters_catalog>

Верни результат строго в формате JSON.
""".strip()
    return base_prompt, catalog_prompt


def request_evaluation(text, labeled_lists):
    evaluator_prompt = f"""
Описание проекта:
<case>
{text}
</case>

Список A:
{json.dumps(labeled_lists["A"], ensure_ascii=False)}

Список B:
{json.dumps(labeled_lists["B"], ensure_ascii=False)}

Какой список выглядит более профессиональным и лучше демонстрирует понимание задачи заказчика, и почему?

Верни результат строго в формате JSON.
""".strip()
    last_error = None
    for attempt in range(1, MAX_API_ATTEMPTS + 1):
        correction = "" if last_error is None else f"\n\nИсправь предыдущий ответ: {last_error}."
        response = client.responses.create(
            model=EVALUATOR_MODEL,
            instructions=EVALUATOR_SYSTEM_PROMPT,
            input=evaluator_prompt + correction,
            text={"format": {"type": "json_object"}},
        )
        try:
            payload = json.loads(response.output_text)
            winner = str(payload.get("winner", "")).strip().upper()
            reason = str(payload.get("reason", "")).strip()
            if winner not in {"A", "B"} or not reason:
                raise ValueError("Нужны winner (A или B) и непустой reason")
            return winner, reason
        except (json.JSONDecodeError, TypeError, ValueError) as error:
            last_error = str(error)
            print(f"Оценка, попытка {attempt}, не прошла проверку: {last_error}")
    raise RuntimeError(f"Не удалось получить корректную оценку: {last_error}")


def exact_binomial_greater_p_value(successes, trials, null_probability=0.5):
    """P(X >= successes) для X ~ Binomial(trials, null_probability)."""
    return sum(
        math.comb(trials, value)
        * null_probability ** value
        * (1 - null_probability) ** (trials - value)
        for value in range(successes, trials + 1)
    )


def save_evaluation_logs(short_rows, detailed_rows):
    EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(short_rows).to_csv(
        EVALUATION_DIR / "short_log.csv", index=False, encoding="utf-8-sig"
    )
    with (EVALUATION_DIR / "detailed_log.jsonl").open("w", encoding="utf-8") as stream:
        for row in detailed_rows:
            stream.write(json.dumps(row, ensure_ascii=False) + "\n")


In [8]:
rng = random.SystemRandom()
short_log = []
detailed_log = []

for test_case_path in case_paths:
    test_case_text = test_case_path.read_text(encoding="utf-8").strip()
    if not test_case_text:
        raise ValueError(f"Файл кейса пуст: {test_case_path}")

    base_prompt, parameter_prompt = make_question_prompts(test_case_text)
    base_questions = request_questions(base_prompt)
    parameter_questions = request_questions(parameter_prompt)

    approaches = ["base", "with_parameters"]
    rng.shuffle(approaches)
    label_to_approach = dict(zip(("A", "B"), approaches))
    questions_by_approach = {
        "base": base_questions,
        "with_parameters": parameter_questions,
    }
    labeled_lists = {
        label: questions_by_approach[approach]
        for label, approach in label_to_approach.items()
    }

    winner_label, evaluator_reason = request_evaluation(test_case_text, labeled_lists)
    winner_approach = label_to_approach[winner_label]
    timestamp = datetime.now(timezone.utc).isoformat()
    short_log.append({
        "timestamp_utc": timestamp,
        "case_id": test_case_path.stem,
        "more_professional": winner_approach,
        "with_parameters_better": winner_approach == "with_parameters",
    })
    detailed_log.append({
        "timestamp_utc": timestamp,
        "case_id": test_case_path.stem,
        "case_file": str(test_case_path),
        "presentation_order": label_to_approach,
        "winner_label": winner_label,
        "winner_approach": winner_approach,
        "evaluator_reason": evaluator_reason,
        "base_questions": base_questions,
        "questions_with_parameters": parameter_questions,
        "generation_model": MODEL,
        "evaluator_model": EVALUATOR_MODEL,
    })
    save_evaluation_logs(short_log, detailed_log)
    print(f"{test_case_path.name}: более профессиональный — {winner_approach} (позиция {winner_label})")

trials = len(short_log)
with_parameters_wins = sum(row["with_parameters_better"] for row in short_log)
p_value = exact_binomial_greater_p_value(with_parameters_wins, trials)
alpha = 0.05
summary = {
    "test_cases": trials,
    "with_parameters_wins": with_parameters_wins,
    "base_wins": trials - with_parameters_wins,
    "with_parameters_win_rate": with_parameters_wins / trials,
    "test": "one-sided exact binomial test, H0: p = 0.5, H1: p > 0.5",
    "p_value": p_value,
    "alpha": alpha,
    "statistically_significant": p_value < alpha,
}
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
(EVALUATION_DIR / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"\nПодход с таблицей лучше в {with_parameters_wins} из {trials} случаев.")
print(f"Односторонний точный биномиальный тест: p-value = {p_value:.6g}.")
print(
    "Результат статистически значим на уровне 0.05."
    if summary["statistically_significant"]
    else "Статистическая значимость на уровне 0.05 не достигнута."
)
display(pd.DataFrame(short_log))
display(pd.Series(summary, name="Результат теста").to_frame())


case_081_zillow_floor_plan_training_models_to_detect_windows_doors_and_openings_in_panora.md: более профессиональный — base (позиция A)
case_082_optimal_drop_times_using_machine_learning.md: более профессиональный — with_parameters (позиция B)
case_085_how_we_used_cross_lingual_transfer_learning_to_categorize_our_content.md: более профессиональный — base (позиция B)
case_086_the_linkedin_generative_ai_application_tech_stack_extending_to_build_ai_agents.md: более профессиональный — with_parameters (позиция A)
case_088_using_ai_to_detect_similar_issues.md: более профессиональный — base (позиция B)
case_091_better_customer_support_using_retrieval_augmented_generation_rag_at_thomson_reut.md: более профессиональный — base (позиция B)
case_092_how_we_use_ai_to_optimize_travel_images.md: более профессиональный — with_parameters (позиция B)
case_093_mercury_agentic_ai_platform_for_llm_powered_recommendation_experiences_at_ebay.md: более профессиональный — base (позиция B)
case_094_a_closer_loo

,timestamp_utc,case_id,more_professional,with_parameters_better
0,2026-09-25T20:28:29.081910+00:00,case_081_zillow_floor_plan_training_models_to_...,base,False
1,2026-09-25T20:29:24.678752+00:00,case_082_optimal_drop_times_using_machine_lear...,with_parameters,True
2,2026-09-25T20:30:20.792668+00:00,case_085_how_we_used_cross_lingual_transfer_le...,base,False
3,2026-09-25T20:31:13.827125+00:00,case_086_the_linkedin_generative_ai_applicatio...,with_parameters,True
4,2026-09-25T20:32:06.508817+00:00,case_088_using_ai_to_detect_similar_issues,base,False
5,2026-09-25T20:32:53.055469+00:00,case_091_better_customer_support_using_retriev...,base,False
6,2026-09-25T20:33:44.091193+00:00,case_092_how_we_use_ai_to_optimize_travel_images,with_parameters,True
7,2026-09-25T20:34:35.052884+00:00,case_093_mercury_agentic_ai_platform_for_llm_p...,base,False
8,2026-09-25T20:35:32.089870+00:00,case_094_a_closer_look_at_the_ai_behind_course...,with_parameters,True
9,2026-09-25T20:36:23.591747+00:00,case_095_introduction_optimising_budget_throug...,with_parameters,True


,Результат теста
test_cases,14
with_parameters_wins,7
base_wins,7
with_parameters_win_rate,0.5
test,"one-sided exact binomial test, H0: p = 0.5, H1..."
p_value,0.604736
alpha,0.05
statistically_significant,False
